# Stock Market Predictor – Exploratory Data Analysis

This notebook walks through downloading historical stock data, visualising price
trends, and inspecting the technical indicators generated by `src/preprocessor.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_fetcher import fetch_stock_data
from src.preprocessor import add_technical_indicators, prepare_features

%matplotlib inline
sns.set_theme(style='darkgrid')

## 1. Download Data

In [ ]:
TICKER = 'AAPL'
PERIOD = '2y'

df = fetch_stock_data(TICKER, period=PERIOD)
print(f'Downloaded {len(df)} rows for {TICKER}')
df.tail()

## 2. Closing Price History

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
df['Close'].plot(ax=ax, title=f'{TICKER} Closing Price ({PERIOD})', ylabel='Price (USD)')
plt.tight_layout()
plt.show()

## 3. Technical Indicators

In [ ]:
df_ind = add_technical_indicators(df)

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Price + moving averages + Bollinger Bands
axes[0].plot(df_ind['Close'], label='Close', linewidth=1)
axes[0].plot(df_ind['SMA_20'], label='SMA 20', linewidth=1)
axes[0].plot(df_ind['SMA_50'], label='SMA 50', linewidth=1)
axes[0].fill_between(df_ind.index, df_ind['BB_Lower'], df_ind['BB_Upper'],
                     alpha=0.1, label='Bollinger Bands')
axes[0].set_title('Price & Moving Averages')
axes[0].legend()

# RSI
axes[1].plot(df_ind['RSI_14'], color='purple', linewidth=1)
axes[1].axhline(70, linestyle='--', color='red', alpha=0.7, label='Overbought (70)')
axes[1].axhline(30, linestyle='--', color='green', alpha=0.7, label='Oversold (30)')
axes[1].set_title('RSI (14)')
axes[1].legend()

# MACD
axes[2].plot(df_ind['MACD'], label='MACD', linewidth=1)
axes[2].plot(df_ind['MACD_Signal'], label='Signal', linewidth=1)
axes[2].axhline(0, linestyle='--', color='grey', alpha=0.5)
axes[2].set_title('MACD')
axes[2].legend()

plt.tight_layout()
plt.show()

## 4. Feature Correlation

In [ ]:
X, y = prepare_features(df_ind)
corr = X.corrwith(y).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
corr.plot(kind='barh', ax=ax, title='Feature Correlation with Target (Next Close)')
plt.tight_layout()
plt.show()